# Single Experiment: Step-by-Step Reproduction

This notebook walks through the full obfuscation pipeline for a single model/dataset combination.
Modify the configuration cell below to reproduce any experiment from the paper.

In [ ]:
import torch
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
# Suppress noisy HTTP logs
for name in ["httpx", "httpcore", "filelock", "huggingface_hub"]:
    logging.getLogger(name).setLevel(logging.WARNING)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## Configuration

Change these settings to run different experiments. Available tasks:
- `classification` — ViT + linear head (CIFAR-10, CIFAR-100, ImageNet, PathMNIST)
- `zero_shot_classification` — CLIP zero-shot
- `object_detection` — YOLOS on COCO
- `zero_shot_object_detection` — OWL-ViT on COCO
- `segmentation` — SegFormer on ADE20K
- `zero_shot_segmentation` — CLIPSeg

In [ ]:
from vit_obfuscation.config.experiment import (
    ExperimentConfig, ModelConfig, ObfuscationConfig,
    EmbeddingTrainingConfig, TaskTrainingConfig, DatasetConfig, EvaluationConfig,
)

config = ExperimentConfig(
    name="vit-cifar10",
    model=ModelConfig(
        hf_model_name_or_path="google/vit-base-patch16-224",
        task="classification",
        num_classes=10,
    ),
    obfuscation=ObfuscationConfig(patch_size=14, group_size=100),
    embedding_training=EmbeddingTrainingConfig(iterations=5000, learning_rate=1e-2, batch_size=32),
    task_training=TaskTrainingConfig(iterations=1000, learning_rate=1e-4, batch_size=256),
    dataset=DatasetConfig(
        hf_dataset_name_or_path="cifar10",
        input_column="img",
        label_column="label",
        train_split="train",
        eval_split="test",
    ),
    evaluation=EvaluationConfig(batch_size=256),
    output_dir="./outputs",
    seed=42,
)

# Or load from YAML:
# config = ExperimentConfig.from_yaml("../configs/experiments/vit_cifar10.yaml")

## Step 1: Load Model and Create Adapter

In [ ]:
from transformers import AutoModel, AutoProcessor
from vit_obfuscation.adapter.model_adapter import ModelAdapter

model = AutoModel.from_pretrained(config.model.hf_model_name_or_path)
processor = AutoProcessor.from_pretrained(config.model.hf_model_name_or_path)

adapter = ModelAdapter(model)
vision_config = adapter.get_vision_config()
print(f"Model type: {model.config.model_type}")
print(f"Embedding spec: {adapter.spec}")
print(f"Image size: {vision_config.image_size}, Patch size: {vision_config.patch_size}, Hidden: {vision_config.hidden_size}")

## Step 2: Create Obfuscation Modules

In [ ]:
from vit_obfuscation.obfuscation.obfuscator import ChannelWisePatchLevelObfuscator
from vit_obfuscation.embedding.embedding import ObfuscationEmbedding

obfuscator = ChannelWisePatchLevelObfuscator(
    image_size=vision_config.image_size,
    num_channels=vision_config.num_channels,
    patch_size=config.obfuscation.patch_size,
    group_size=config.obfuscation.group_size,
)

obf_embedding = ObfuscationEmbedding(
    image_size=vision_config.image_size,
    num_channels=vision_config.num_channels,
    patch_size=vision_config.patch_size,
    embed_dim=vision_config.hidden_size,
    num_extra_tokens=len(adapter.spec.extra_tokens),
)

adapter.copy_frozen_params(obf_embedding)
print(f"Obfuscator params: {sum(p.numel() for p in obfuscator.buffers()):,} (non-trainable)")
print(f"Embedding params: {sum(p.numel() for p in obf_embedding.parameters()):,} (trainable)")

## Step 3: Visualize Obfuscation

Load a sample image and show the effect of obfuscation.

In [ ]:
import matplotlib.pyplot as plt
from datasets import load_dataset

sample_dataset = load_dataset(config.dataset.hf_dataset_name_or_path, split=config.dataset.eval_split)
sample_image = sample_dataset[0][config.dataset.input_column]
if hasattr(sample_image, 'convert'):
    sample_image = sample_image.convert("RGB")

inputs = processor(images=sample_image, return_tensors="pt")
pixel_values = inputs["pixel_values"]
obfuscated = obfuscator(pixel_values)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

clean_img = pixel_values[0].permute(1, 2, 0).numpy()
clean_img = (clean_img - clean_img.min()) / (clean_img.max() - clean_img.min())
axes[0].imshow(clean_img)
axes[0].set_title("Clean (preprocessed)")
axes[0].axis("off")

obf_img = obfuscated[0].permute(1, 2, 0).numpy()
obf_img = (obf_img - obf_img.min()) / (obf_img.max() - obf_img.min())
axes[1].imshow(obf_img)
axes[1].set_title("Obfuscated")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## Step 4: Train Deobfuscation Embedding

This trains the patch embedding to reconstruct original ViT embeddings from obfuscated images.
Uses ImageNet-1k-256 as proxy dataset (task-agnostic).

In [ ]:
from vit_obfuscation.runner.embedding_trainer import EmbeddingTrainer
from vit_obfuscation.runner.runner import save_checkpoint, load_checkpoint, _checkpoint_key

ckpt_key = _checkpoint_key(config)

if load_checkpoint(obfuscator, obf_embedding, config.output_dir, ckpt_key):
    print("Loaded existing checkpoint \u2014 skipping training")
else:
    trainer = EmbeddingTrainer(
        adapter=adapter,
        obfuscator=obfuscator,
        obf_embedding=obf_embedding,
        processor=processor,
        config=config.embedding_training,
    )
    trainer.train()
    obf_embedding = trainer.obf_embedding
    obfuscator = trainer.obfuscator
    adapter = ModelAdapter(trainer.adapter.model, adapter.spec)
    save_checkpoint(obfuscator, obf_embedding, config.output_dir, ckpt_key)
    print("Training complete \u2014 checkpoint saved")

## Step 5: Run Task and Evaluate

In [ ]:
from vit_obfuscation.runner.runner import _import_task_class

TaskClass = _import_task_class(config.model.task)
task = TaskClass(
    adapter=adapter,
    obfuscator=obfuscator,
    obf_embedding=obf_embedding,
    processor=processor,
    config=config,
)

print("Evaluating clean baseline...")
clean_results = task.evaluate(with_obfuscation=False)
print(f"Clean: {dict(clean_results.items())}")

if config.task_training is not None:
    print("\nTraining task head...")
    task.train_task()

print("\nEvaluating with obfuscation...")
obf_results = task.evaluate(with_obfuscation=True)
print(f"Obfuscated: {dict(obf_results.items())}")

## Results Summary

In [ ]:
import json, os

results = {
    "experiment": config.name,
    "model": config.model.hf_model_name_or_path,
    "task": config.model.task,
    "clean": dict(clean_results.items()),
    "obfuscated": dict(obf_results.items()),
}

os.makedirs(config.output_dir, exist_ok=True)
path = os.path.join(config.output_dir, f"{config.name}_results.json")
with open(path, "w") as f:
    json.dump(results, f, indent=2, default=str)

print(f"Results saved to {path}")
print(json.dumps(results, indent=2, default=str))